<h1> Clean data </h1>

In [1]:
######### DELETE REDUCTION STATS IF NEEDED, AS OTHERWISE NEW ROWS WILL BE APPENDED INSTEAD OF OVERWRITE #########

In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import clean_data_helper

BASE_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\English")
OUTPUT_ROOT = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Clean_Data")

MASKS_DIR = OUTPUT_ROOT / "cleaning_masks"
STATS_OUTPUT_PATH = OUTPUT_ROOT / "detailed_reduction_stats.csv"

MASKS_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2013 # 1951
END_YEAR = 2013 # 2023

In [2]:
if __name__ == "__main__":
    
    target_years = range(START_YEAR, END_YEAR + 1)
    all_files = [f for y in target_years for f in (BASE_DIR / str(y)).glob("subset_*.parquet")]

    num_workers = 8
    print(f"Processing {len(all_files)} files with {num_workers} workers...")
    print(f"Saving output to: {OUTPUT_ROOT}")

    results = []
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        
        # Pass MASKS_DIR as a string to the worker
        futures = {executor.submit(clean_data_helper.process_single_file, f, str(MASKS_DIR)): f for f in all_files}
        
        for future in tqdm(as_completed(futures), total=len(all_files)):
            res = future.result()
            results.append(res)
            if "error" not in res:
                # Stats CSV will now be saved in ...\Data\Clean_Data\
                pd.DataFrame([res]).to_csv(STATS_OUTPUT_PATH, mode='a', index=False, 
                                          header=not os.path.exists(STATS_OUTPUT_PATH))

            else:
                # Error path: Print specific failure details to the console
                print(f"\n[!] FAILED: {res.get('file')} | Error: {res.get('error')}")

    print(f"\nProcessing complete. Files saved in {OUTPUT_ROOT}")

Processing 100 files with 8 workers...
Saving output to: C:\Users\danielyoon\Dropbox\hist_LLM\Data\Clean_Data


  0%|          | 0/100 [00:00<?, ?it/s]


Processing complete. Files saved in C:\Users\danielyoon\Dropbox\hist_LLM\Data\Clean_Data


<h1> Sanity Check </h1>

In [18]:
import pandas as pd
import re
from pathlib import Path

# --- CONFIG FOR DEBUGGING ---
# Pick one specific file to audit
DEBUG_FILE = r"C:\Users\danielyoon\Dropbox\hist_LLM\English\1900\subset_2.parquet"
MASK_FILE = r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Clean_Data\cleaning_masks\1900\subset_2_mask.parquet"

def fast_punct(text):
    if not text: return 0
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if not lines: return 0
    ends = sum(1 for l in lines if l.endswith(('.', '?', '!')))
    return ends / len(lines)

def run_sanity_check(file_path):
    print(f"--- Auditing File: {Path(file_path).name} ---")
    
    # 1. Load Data
    df = pd.read_parquet(file_path, columns=['text'])
    df['original_index'] = df.index # Explicitly track the range index
    
    # Stage 1: Symbol Ratio
    sym_counts = df['text'].str.count(r'[^\w\s]')
    word_counts = df['text'].str.split().str.len()
    df['fail_sym'] = (sym_counts / (word_counts + 1e-6)) > 0.25

    # Stage 2: Punctuation
    df['fail_punct'] = df['text'].apply(fast_punct) < 0.12
    
    # Stage 3: Stopwords
    stop_pattern = r'\b(the|and|is|of|to|it|that|in)\b'
    stop_counts = df['text'].str.count(stop_pattern, flags=re.IGNORECASE)
    df['fail_stops'] = stop_counts < 3
    
    # Final Clean Flag
    df['is_clean'] = ~(df['fail_sym'] | df['fail_punct'] | df['fail_stops'])
    
    # --- VISUAL VERIFICATION ---
    print("\n[REJECTION SAMPLE] Showing 5 rows that failed the filter:")
    display_cols = ['original_index', 'fail_sym', 'fail_punct', 'fail_stops', 'text']
    print(df[~df['is_clean']][display_cols].head(5))

    print("\n[CLEAN SAMPLE] Showing 5 rows that passed:")
    print(df[df['is_clean']][display_cols].head(5))
    
    # --- MASK INTEGRITY CHECK ---
    if Path(MASK_FILE).exists():
        mask_df = pd.read_parquet(MASK_FILE)
        clean_ids_in_memory = df[df['is_clean']]['original_index'].tolist()
        ids_in_saved_mask = mask_df['original_index'].tolist()
        
        matches = clean_ids_in_memory == ids_in_saved_mask
        print(f"\n[INTEGRITY] Does the saved mask match the logic? {matches}")
        print(f"Clean rows identified: {len(clean_ids_in_memory)}")
    else:
        print("\n[NOTICE] Mask file not found. Run your Step 1 code first to generate it.")

    return df

In [19]:
df_check = run_sanity_check(DEBUG_FILE)
df_oiginal = pd.read_parquet(DEBUG_FILE)
df_mask = pd.read_parquet(MASK_FILE)

--- Auditing File: subset_2.parquet ---

[REJECTION SAMPLE] Showing 5 rows that failed the filter:
   original_index  fail_sym  fail_punct  fail_stops  \
0               0     False        True       False   
3               3      True       False       False   
4               4      True       False       False   
5               5      True       False       False   
7               7      True       False       False   

                                                text  
0  SUPPLEMENT TO The Ely Miner. Ely, Minn. FRIDAY...  
3  NEW HAVEN MORNING JOURNAL AND COURIER, SATURDA...  
4  THE WAR. The Queen's Present Saves a Life. A c...  
5  TIMARU, CANTERBURY: MONDAY, DECEMBER 10, 1900....  
7  Headache Blliouaness, sour stomsch, conatlpa t...  

[CLEAN SAMPLE] Showing 5 rows that passed:
    original_index  fail_sym  fail_punct  fail_stops  \
1                1     False       False       False   
2                2     False       False       False   
6                6     False

In [20]:
df_oiginal[["text"]]

,text
0,"SUPPLEMENT TO The Ely Miner. Ely, Minn. FRIDAY..."
1,"WELLINGTON NOTES. (BY ""THE VAGABOND."") ""Buboni..."
2,"She saUy Jiyltt. FRIDAY, JUNE 15, 1900. Headac..."
3,"NEW HAVEN MORNING JOURNAL AND COURIER, SATURDA..."
4,THE WAR. The Queen's Present Saves a Life. A c...
...,...
6186,"Volume XXXV 9776 OTAGO, N.Z. North Otago Times..."
6187,Free Messet It costs you advancement The follo...
6188,ELECTION NOTICI —BY THE— Board of County Commi...
6189,"Mexico"" Missouri Message. VOLUME 1. MEXICO, AU..."


In [22]:
temp = df_oiginal[["text"]]
print(temp[temp.index == 29]["text"].values[0])

With direct legislation as the paramount issue, nominate William J. Bryan (and the assurance has been given by Mr. Bryan's friends that he will accept the nomination) and some Southern Populist; that the free Silver Republicans will endorse the platform and its candidates; that Mr. Bryan's friends will go before the National Democratic Convention with a demand for the endorsement of the Populist Convention, and it is argued that they will not refuse. In the event that they do, Mr. Bryan may refuse the nomination at the hands of the Democracy. The action of itself will, by the Democrats, signal their utter defeat and BROADWAY Happy Man! He knows Harris' $2.95 Shoes. 520 Pine CORNER LOCUST ST. of this pattern would decorate the dinner table! It will last a lifetime. What a beautiful, useful and highly prized wedding gift it would make! $2.50 Each. CORNER OF LOCUST ST. also the Populist at will be at the door of the Democratic Convention. "We haven't this time, as far as these proposition

In [23]:
df_check[(df_check["fail_stops"] == True)&(df_check["fail_sym"] != True)]

,text,original_index,fail_sym,fail_punct,fail_stops,is_clean


In [11]:
df_check

,text,original_index,fail_sym,fail_punct,fail_stops,is_clean
0,"All Over the State, CARON The County Correspon...",0,True,False,False,False
1,"OTAGO WITNESS. November 7, 1900 If so, why did...",1,True,False,False,False
2,"THE REPUBLIC: SATURDAY, JULY 14, 1900. WHEAT A...",2,True,False,False,False
3,In 1854 Bishop Kemper was for the second time ...,3,False,False,False,True
4,Shoes for Hot Weather Ladies' House Slippers h...,4,True,False,False,False
...,...,...,...,...,...,...
6129,BOERS ARE AT BLOEMFONTEIN. They Have Left Old ...,6129,True,False,False,False
6130,Marietta Daily Leader THE ONLY ASSOCIATED PRES...,6130,False,False,False,True
6131,A men PROSPECT. The mining and milling company...,6131,False,False,False,True
6132,By this time you are charging madly around the...,6132,False,False,False,True


In [12]:
df_mask

,original_index
0,3
1,6
2,7
3,8
4,11
...,...
3393,6126
3394,6130
3395,6131
3396,6132
